In [7]:
!pip -q install faiss-cpu sentence-transformers transformers accelerate

# Imports

In [8]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline, AutoTokenizer

# Load Dataset

In [9]:
train = pd.read_csv("/content/train (1).csv")

print(train.shape)
train.head()

(2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [10]:
kb = []

for _, row in train.iterrows():
    correct = row["answer"]
    kb.append(str(row[correct]))

# Build Knowledge Base

# Load Models

In [11]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

## Build FAISS Index

In [12]:
kb_embeddings = embed_model.encode(
    kb,
    show_progress_bar=True,
    convert_to_numpy=True
)

index = faiss.IndexFlatL2(kb_embeddings.shape[1])

index.add(kb_embeddings)

print(index.ntotal)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

2000


# Helper Function

In [13]:
def retrieve(prompt, k=5):
    emb = embed_model.encode(
        [prompt],
        convert_to_numpy=True
    )

    D, I = index.search(emb, k)

    return I[0], D[0]

# Q1

In [14]:
row = train.iloc[150]

prompt = str(row["prompt"])

labels = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"])
]

correct_text = str(row[row["answer"]])

result = zs(prompt, labels)

score = dict(zip(result["labels"], result["scores"]))

print("Q1 =", round(score[correct_text],3))

Q1 = 0.384


# Q2

In [15]:
retrieved, _ = retrieve(prompt,10)

rank = np.where(retrieved==150)[0]

if len(rank)==0:
    print("Not in Top10")
else:
    print("Q2 =", rank[0]+1)

Q2 = 10


# Q3

In [16]:
docs = [kb[i] for i in retrieved]

pairs = [[prompt,d] for d in docs]

scores = cross_encoder.predict(pairs)

order = np.argsort(scores)[::-1]

reranked = retrieved[order]

rank = np.where(reranked==150)[0]

print("Q3 =", rank[0]+1)

Q3 = 1


# Q4

In [17]:
row = train.iloc[42]

prompt = str(row["prompt"])

retrieved,_ = retrieve(prompt,5)

docs = [kb[i] for i in retrieved]

context = " ".join(docs)

text = f"Context: {context} Question: {prompt}"

tokens = tokenizer(
    text,
    truncation=False
)

print("Q4 =", len(tokens["input_ids"]))

Q4 = 216


# Q5

In [18]:
row = train.iloc[150]

prompt = str(row["prompt"])

true_doc = kb[150]

labels = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"])
]

correct = str(row[row["answer"]])

rag = f"Context: {true_doc} Question: {prompt}"

result = zs(rag,labels)

score = dict(zip(result["labels"],result["scores"]))

print("Q5 =", round(score[correct],3))

Q5 = 0.989


# Q6

In [19]:
wrong_doc = kb[999]

rag = f"Context: {wrong_doc} Question: {prompt}"

result = zs(rag,labels)

score = dict(zip(result["labels"],result["scores"]))

print("Q6 =", round(score[correct],3))

Q6 = 0.529


# Q7

In [20]:
hits = 0

for i in range(100):

    row = train.iloc[i]

    prompt = str(row["prompt"])

    correct = str(row[row["answer"]])

    retrieved,_ = retrieve(prompt,5)

    docs = [kb[j] for j in retrieved]

    hit = any(correct in d for d in docs)

    if hit:
        hits += 1

rate = hits/100*100

print("Q7 =", round(rate,1))

Q7 = 73.0


In [21]:
def map3(pred,true):

    if true in pred:

        return 1/(pred.index(true)+1)

    return 0

# Q8


In [22]:
scores=[]

letters=["A","B","C","D","E"]

for i in range(20):

    row=train.iloc[i]

    prompt=str(row["prompt"])

    retrieved,_=retrieve(prompt,5)

    docs=[kb[j] for j in retrieved]

    pairs=[[prompt,d] for d in docs]

    ce_scores=cross_encoder.predict(pairs)

    best_doc=docs[np.argmax(ce_scores)]

    rag=f"Context: {best_doc} Question: {prompt}"

    labels=[
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    result=zs(rag,labels)

    score_dict=dict(zip(result["labels"],result["scores"]))

    option_scores=[]

    for l in letters:

        option_scores.append(
            score_dict[str(row[l])]
        )

    order=np.argsort(option_scores)[::-1]

    pred=[letters[j] for j in order[:3]]

    truth=row["answer"]

    scores.append(map3(pred,truth))

print("Q8 =", round(np.mean(scores),3))

Q8 = 0.975
